In [14]:
import json
import yt_dlp

URL = 'https://www.youtube.com/@furkanhoroz3302/videos'


# ℹ️ See help(yt_dlp.YoutubeDL) for a list of available options and public functions
ydl_opts = {}
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(URL, download=False)

    # ℹ️ ydl.sanitize_info makes the info json-serializable
    print(json.dumps(ydl.sanitize_info(info)))

[youtube:tab] Extracting URL: https://www.youtube.com/@furkanhoroz3302/videos
[youtube:tab] @furkanhoroz3302/videos: Downloading webpage
[download] Downloading playlist: Furkan Horoz - Videos
[youtube:tab] Playlist Furkan Horoz - Videos: Downloading 4 items of 4
[download] Downloading item 1 of 4
[youtube] Extracting URL: https://www.youtube.com/watch?v=LJUWaamrJmU
[youtube] LJUWaamrJmU: Downloading webpage
[youtube] LJUWaamrJmU: Downloading tv client config
[youtube] LJUWaamrJmU: Downloading tv player API JSON
[youtube] LJUWaamrJmU: Downloading ios player API JSON
[youtube] LJUWaamrJmU: Downloading m3u8 information
[download] Downloading item 2 of 4
[youtube] Extracting URL: https://www.youtube.com/watch?v=2FMY1TlTN9c
[youtube] 2FMY1TlTN9c: Downloading webpage
[youtube] 2FMY1TlTN9c: Downloading tv client config
[youtube] 2FMY1TlTN9c: Downloading tv player API JSON
[youtube] 2FMY1TlTN9c: Downloading ios player API JSON
[youtube] 2FMY1TlTN9c: Downloading m3u8 information
[download] Down

In [15]:
len(info)

27

In [16]:
info['entries'][0]

{'id': 'LJUWaamrJmU',
 'title': 'sakar Şakir merhaba arkadaşlar ben geldim 😂',
 'formats': [{'format_id': 'sb2',
   'format_note': 'storyboard',
   'ext': 'mhtml',
   'protocol': 'mhtml',
   'acodec': 'none',
   'vcodec': 'none',
   'url': 'https://i.ytimg.com/sb/LJUWaamrJmU/storyboard3_L0/default.jpg?sqp=-oaymwENSDfyq4qpAwVwAcABBqLzl_8DBgifg-COBg==&sigh=rs$AOn4CLBQqz7X7RIyxniAs77kTpoPOv_m0Q',
   'width': 48,
   'height': 27,
   'fps': 2.6315789473684212,
   'rows': 10,
   'columns': 10,
   'fragments': [{'url': 'https://i.ytimg.com/sb/LJUWaamrJmU/storyboard3_L0/default.jpg?sqp=-oaymwENSDfyq4qpAwVwAcABBqLzl_8DBgifg-COBg==&sigh=rs$AOn4CLBQqz7X7RIyxniAs77kTpoPOv_m0Q',
     'duration': 38.0}],
   'audio_ext': 'none',
   'video_ext': 'none',
   'vbr': 0,
   'abr': 0,
   'tbr': None,
   'resolution': '48x27',
   'aspect_ratio': 1.78,
   'filesize_approx': None,
   'http_headers': {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.451

In [18]:
# access to first video attributes
video = info['entries'][0]
print(video['title'])
print(video['id'])
print(video['description'])
print(video['original_url'])
print(video['view_count'])
print(video['like_count'])
print(video['duration'])
print(video["formats"][0]['fragments'][0]["duration"])  # duration in seconds
print(video["duration_string"])  # duration in string
print(video["playlist_id"]) 
print(video["playlist_uploader"])  # playlist title

sakar Şakir merhaba arkadaşlar ben geldim 😂
LJUWaamrJmU

https://www.youtube.com/watch?v=LJUWaamrJmU
5105
43
38
38.0
38
UChhwsLweKP6bnUCObNBNPzw
Furkan Horoz


# Function

In [4]:
import json
import yt_dlp
import pandas as pd

def get_channel_videos_metadata(channel_url):
    """
    Extracts metadata for all videos in a given YouTube channel URL
    and returns it as a Pandas DataFrame.

    Args:
        channel_url (str): The URL of the YouTube channel.

    Returns:
        pandas.DataFrame: A DataFrame containing video metadata, or None if an error occurs.
                          Returns an empty DataFrame if no video data is collected.
    """
    ydl_opts = {
        'extract_flat': 'in_playlist',  # Faster for just getting video entries in a playlist/channel
        'quiet': True,                  # Suppress console output from yt_dlp
        'force_generic_extractor': False,
        'rejecttitle': '#shorts'        # Option to ignore YouTube Shorts
    }

    videos_data = []

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            # Extract basic info for all entries in the channel/playlist
            # This will give a list of video entries (dictionaries)
            # channel_info itself is a dictionary containing metadata about the channel/playlist
            channel_info = ydl.extract_info(channel_url, download=False, process=True)

            # Extract playlist-level (channel-level) information once
            # The 'id' of the channel_info is the playlist ID (channel ID in this case)
            # The 'uploader' or 'channel' is the name of the channel (playlist uploader)
            playlist_id_from_channel = channel_info.get('id')
            # 'channel' usually holds the channel name, 'uploader' can also be used.
            playlist_uploader_from_channel = channel_info.get('channel') or channel_info.get('uploader')


            if 'entries' in channel_info and channel_info['entries']:
                # Now, for each video entry, extract detailed information
                for entry in channel_info['entries']:
                    # Ensure entry is not None and has a URL
                    if entry and entry.get('url'):
                        video_url = entry['url']
                        print(f"Processing video URL: {video_url}") # Log which video URL is being processed

                        # It's more reliable to extract info for each video individually
                        # to get all desired fields like like_count, view_count etc.
                        video_info_opts = {
                            'quiet': True,
                            'extract_flat': False # Ensure we get full metadata for individual video
                            }
                        try:
                            with yt_dlp.YoutubeDL(video_info_opts) as video_ydl:
                                video_details = video_ydl.extract_info(video_url, download=False)

                                title = video_details.get('title')
                                video_id = video_details.get('id')
                                description = video_details.get('description')
                                original_url = video_details.get('webpage_url')
                                view_count = video_details.get('view_count')
                                like_count = video_details.get('like_count')
                                duration_seconds = video_details.get('duration')
                                duration_string = video_details.get('duration_string')

                                videos_data.append({
                                    'title': title,
                                    'id': video_id,
                                    'description': description,
                                    'original_url': original_url,
                                    'view_count': view_count,
                                    'like_count': like_count,
                                    'duration_seconds': duration_seconds,
                                    'duration_string': duration_string,
                                    'playlist_id': playlist_id_from_channel, # Added
                                    'playlist_uploader': playlist_uploader_from_channel # Added
                                })
                                print(f"Successfully processed: {title}")
                        except yt_dlp.utils.DownloadError as e_video_dl:
                            print(f"yt-dlp error processing video {video_url}: {e_video_dl}")
                            videos_data.append({
                                'title': entry.get('title', 'N/A (DownloadError)'),
                                'id': entry.get('id', 'N/A'),
                                'description': f"Error fetching full details: {e_video_dl}",
                                'original_url': video_url,
                                'view_count': None,
                                'like_count': None,
                                'duration_seconds': entry.get('duration'),
                                'duration_string': None,
                                'playlist_id': playlist_id_from_channel, # Added
                                'playlist_uploader': playlist_uploader_from_channel # Added
                            })
                        except Exception as e_video:
                            print(f"Generic error processing video {video_url}: {e_video}")
                            videos_data.append({
                                'title': entry.get('title', 'N/A (Exception)'),
                                'id': entry.get('id', 'N/A'),
                                'description': f"Error fetching full details: {e_video}",
                                'original_url': video_url,
                                'view_count': None,
                                'like_count': None,
                                'duration_seconds': entry.get('duration'),
                                'duration_string': None,
                                'playlist_id': playlist_id_from_channel, # Added
                                'playlist_uploader': playlist_uploader_from_channel # Added
                            })
                    elif entry and not entry.get('url'):
                        print(f"Skipping entry without URL: {entry.get('title', 'Unknown title')}")
                    elif not entry:
                        print("Skipping a None entry in channel_info['entries']")
            else:
                print(f"No video entries found for URL: {channel_url}. Channel info: {channel_info}")
                return pd.DataFrame()

    except yt_dlp.utils.DownloadError as e:
        print(f"yt-dlp download error for channel {channel_url}: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred for channel {channel_url}: {e}")
        return None

    if not videos_data:
        print("No data was collected for any video from the channel.")
        return pd.DataFrame()

    df = pd.DataFrame(videos_data)
    return df

# Example of how to use the function (optional, can be in a separate script):
if __name__ == '__main__':
     # --- IMPORTANT ---
    # Replace with the actual YouTube channel URL you want to process.
    CHANNEL_URL = 'https://engelsiz.kanald.com.tr/esref-ruya/bolumler/esref-ruya-ilk-bolum' 

    print(f"Fetching videos from channel: {CHANNEL_URL}")
    video_df = get_channel_videos_metadata(CHANNEL_URL)

    if video_df is not None and not video_df.empty:
        print("\n--- Video Metadata DataFrame ---")
         # Display all columns to verify new additions
        pd.set_option('display.max_columns', None)
#
        print("\n--- DataFrame Info ---")
        video_df.info()
#
         # To save to CSV:
        video_df.to_csv('/Users/halilergul/Desktop/personal_projects/voice-data/meta_datasets/engelsiz_kanalD_videos_metadata.csv', index=False, encoding='utf-8')
        print("\nDataFrame saved to channel_videos_metadata.csv")
    elif video_df is not None and video_df.empty:
        print("No videos were processed successfully, or the channel has no videos. DataFrame is empty.")
    else:
        print("Failed to generate video metadata DataFrame due to an error during channel processing.")



Fetching videos from channel: https://engelsiz.kanald.com.tr/esref-ruya/bolumler/esref-ruya-ilk-bolum


ERROR: Unsupported URL: https://engelsiz.kanald.com.tr/esref-ruya/bolumler/esref-ruya-ilk-bolum


yt-dlp download error for channel https://engelsiz.kanald.com.tr/esref-ruya/bolumler/esref-ruya-ilk-bolum: ERROR: Unsupported URL: https://engelsiz.kanald.com.tr/esref-ruya/bolumler/esref-ruya-ilk-bolum
Failed to generate video metadata DataFrame due to an error during channel processing.
